In [1]:
from pyspark.sql import functions as F

assess = (
    spark.table("silver_assessments")
    .withColumn("week_number", F.floor(F.col("date") / 7) + 1)
    .withColumn("is_pass", F.when(F.col("score_filled") >= 40, 1).otherwise(0))
)

vle = (
    spark.table("silver_vle_activity")
    .withColumn("week_number", F.floor(F.col("date") / 7) + 1)
)

risk = spark.table("silver_students").select(
    "student_id", "course_id", "semester", "is_at_risk"
)

weekly_clicks = (
    vle.groupBy("course_id", "semester", "week_number")
    .agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("student_id").alias("active_students"),
        F.avg("sum_click").alias("avg_clicks_per_student")
    )
)

weekly_assess = (
    assess.groupBy("course_id", "semester", "week_number")
    .agg(
        F.avg("score_filled").alias("avg_score"),
        F.avg("submission_delay_days").alias("avg_submission_delay"),
        F.sum("is_pass").alias("passed_assessments"),
        F.count("*").alias("total_assessments")
    )
    .withColumn(
        "pass_rate",
        F.when(
            F.col("total_assessments") > 0,
            F.col("passed_assessments") / F.col("total_assessments")
        ).otherwise(None)
    )
)

weekly_risk = (
    risk.join(
        vle.select("student_id", "course_id", "semester", "week_number").distinct(),
        ["student_id", "course_id", "semester"], "left"
    )
    .groupBy("course_id", "semester", "week_number")
    .agg(F.sum("is_at_risk").alias("risk_count"))
)

weekly_trends = (
    weekly_clicks
    .join(weekly_assess, ["course_id", "semester", "week_number"], "full")
    .join(weekly_risk,   ["course_id", "semester", "week_number"], "full")
    .fillna(0, subset=[
        "total_clicks", "active_students", "avg_clicks_per_student",
        "avg_score", "avg_submission_delay", "passed_assessments",
        "total_assessments", "risk_count"
    ])
    .withColumn("_computed_at", F.current_timestamp())
    .select(
        "course_id", "semester", "week_number",
        "total_clicks", "active_students", "avg_clicks_per_student",
        "avg_score", "avg_submission_delay",
        "passed_assessments", "total_assessments", "pass_rate",
        "risk_count", "_computed_at"
    )
)

weekly_trends.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_weekly_trends")

print(f"✅ gold_weekly_trends: {weekly_trends.count():,} rows")
weekly_trends.orderBy("course_id", "week_number").show(5)

StatementMeta(, ff0d4830-ff72-4091-9ec2-2686f2a3798d, 3, Finished, Available, Finished, False)

✅ gold_weekly_trends: 904 rows
+---------+--------+-----------+------------+---------------+----------------------+---------+--------------------+------------------+-----------------+---------+----------+--------------------+
|course_id|semester|week_number|total_clicks|active_students|avg_clicks_per_student|avg_score|avg_submission_delay|passed_assessments|total_assessments|pass_rate|risk_count|        _computed_at|
+---------+--------+-----------+------------+---------------+----------------------+---------+--------------------+------------------+-----------------+---------+----------+--------------------+
|      AAA|   2014J|       NULL|           0|              0|                   0.0|      0.0|                 0.0|                 0|                0|     NULL|         6|2026-04-20 06:22:...|
|      AAA|   2013J|       NULL|           0|              0|                   0.0|      0.0|                 0.0|                 0|                0|     NULL|         5|2026-04-20 06:22

In [2]:
pred = spark.table("gold_student_risk_predictions")

alerts = (
    pred
    .withColumn(
        "alert_level",
        F.when(F.col("risk_probability") >= 0.8, "CRITICAL")
         .when(F.col("risk_probability") >= 0.6, "HIGH")
         .when(F.col("risk_probability") >= 0.4, "MEDIUM")
         .otherwise("LOW")
    )
    .withColumn(
        "alert_reason",
        F.when(F.col("risk_probability") >= 0.8, "Very high probability of academic failure")
         .when(F.col("risk_probability") >= 0.6, "High probability of academic risk")
         .when(F.col("risk_probability") >= 0.4, "Moderate academic risk")
         .otherwise("No immediate action required")
    )
    .select(
        "student_id", "avg_score", "total_clicks",
        "num_of_prev_attempts", "studied_credits",
        "is_at_risk", "predicted_at_risk", "risk_probability",
        "alert_level", "alert_reason",
        "_model_version", "_scored_at"
    )
)

alerts.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_alerts")

print(f"✅ gold_alerts: {alerts.count():,} rows")
alerts.groupBy("alert_level").count().orderBy("alert_level").show()

StatementMeta(, ff0d4830-ff72-4091-9ec2-2686f2a3798d, 4, Finished, Available, Finished, False)

✅ gold_alerts: 28,785 rows
+-----------+-----+
|alert_level|count|
+-----------+-----+
|   CRITICAL| 7054|
|       HIGH| 4025|
|        LOW|11458|
|     MEDIUM| 6248|
+-----------+-----+



In [3]:
risk_base = spark.table("gold_student_risk")

enriched = (
    spark.table("gold_student_risk_predictions")
    .join(
        risk_base.select("student_id", "course_id", "semester", "final_result", "risk_band"),
        on="student_id",
        how="left"
    )
)

enriched.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_student_risk_predictions")

print(f"✅ gold_student_risk_predictions enriched: {enriched.count():,} rows")
enriched.printSchema()

StatementMeta(, ff0d4830-ff72-4091-9ec2-2686f2a3798d, 5, Finished, Available, Finished, False)

✅ gold_student_risk_predictions enriched: 28,785 rows
root
 |-- student_id: integer (nullable = true)
 |-- avg_score: double (nullable = true)
 |-- total_clicks: long (nullable = true)
 |-- num_of_prev_attempts: integer (nullable = true)
 |-- studied_credits: integer (nullable = true)
 |-- is_at_risk: integer (nullable = true)
 |-- predicted_at_risk: integer (nullable = true)
 |-- risk_probability: double (nullable = true)
 |-- _model_version: string (nullable = true)
 |-- _scored_at: timestamp (nullable = true)
 |-- course_id: string (nullable = true)
 |-- semester: string (nullable = true)
 |-- final_result: string (nullable = true)
 |-- risk_band: string (nullable = true)



In [4]:
pred = spark.table("gold_student_risk_predictions")

recommendations = pred.select(
    "student_id", "course_id", "semester",
    "risk_probability", "avg_score", "total_clicks", "num_of_prev_attempts"
).withColumn(
    "rec_1",
    F.when(F.col("risk_probability") >= 0.6, "Immediate academic support intervention required")
     .otherwise(None)
).withColumn(
    "rec_2",
    F.when(F.col("avg_score") < 50, "Enroll in remedial assessment sessions")
     .otherwise(None)
).withColumn(
    "rec_3",
    F.when(F.col("total_clicks") < 500, "Increase VLE engagement — review materials weekly")
     .otherwise(None)
).withColumn(
    "rec_4",
    F.when(F.col("num_of_prev_attempts") >= 2, "Meet academic advisor to review study strategy")
     .otherwise(None)
).withColumn(
    "primary_recommendation",
    F.coalesce(F.col("rec_1"), F.col("rec_2"), F.col("rec_3"), F.col("rec_4"), 
               F.lit("Maintain current study habits — student is on track"))
).withColumn("_computed_at", F.current_timestamp()) \
 .select(
    "student_id", "course_id", "semester",
    "risk_probability", "primary_recommendation",
    "rec_1", "rec_2", "rec_3", "rec_4",
    "_computed_at"
)

recommendations.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_recommendations")

print(f"✅ gold_recommendations: {recommendations.count():,} rows")
recommendations.groupBy("primary_recommendation").count().orderBy("count", ascending=False).show(truncate=False)

StatementMeta(, ff0d4830-ff72-4091-9ec2-2686f2a3798d, 6, Finished, Available, Finished, False)

✅ gold_recommendations: 28,785 rows
+---------------------------------------------------+-----+
|primary_recommendation                             |count|
+---------------------------------------------------+-----+
|Maintain current study habits — student is on track|13936|
|Immediate academic support intervention required   |11079|
|Increase VLE engagement — review materials weekly  |3573 |
|Meet academic advisor to review study strategy     |135  |
|Enroll in remedial assessment sessions             |62   |
+---------------------------------------------------+-----+



In [5]:
gold_tables = [
    "gold_course_difficulty",
    "gold_student_risk",
    "gold_instructor_effectiveness",
    "gold_engagement",
    "gold_weekly_trends",
    "gold_student_risk_predictions",
    "gold_alerts",
    "gold_recommendations",
]

print("--- Final Gold Layer ---")
for table in gold_tables:
    count = spark.table(table).count()
    print(f"✅ {table}: {count:,} rows")

StatementMeta(, ff0d4830-ff72-4091-9ec2-2686f2a3798d, 7, Finished, Available, Finished, False)

--- Final Gold Layer ---
✅ gold_course_difficulty: 7 rows
✅ gold_student_risk: 28,785 rows
✅ gold_instructor_effectiveness: 7 rows
✅ gold_engagement: 394 rows
✅ gold_weekly_trends: 904 rows
✅ gold_student_risk_predictions: 28,785 rows
✅ gold_alerts: 28,785 rows
✅ gold_recommendations: 28,785 rows
